# HMR-GNN — Run on Kaggle (background, no tab-sitting)

**Heterophily-Aware Multi-Relational GNN for Robust Fake Account Detection.**

Unlike Colab, Kaggle can run this **in the background**: use **Save Version -> Save & Run All
(Commit)** and you can close the tab and multitask. The run finishes on Kaggle's servers and
the results are saved as the notebook's Output (downloadable).

### One-time setup (right panel -> Settings / Session options)
1. **Accelerator -> GPU T4 x2** (or P100).
2. **Internet -> On**  (needed to `git clone`; requires a phone-verified Kaggle account).
3. Then click **Save Version -> Save & Run All (Commit)** to run it in the background.

Results land in `/kaggle/working/results` and are bundled into `hmrgnn_results.zip`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Clone the repository (code + data)
Requires **Internet -> On** in the session settings.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/HMR-GNN'):
    !git clone https://github.com/Vincentfernandes17/HMR-GNN.git
os.chdir('/kaggle/working/HMR-GNN')
!git pull
assert os.path.exists('data/MGTAB/features.pt'), 'MGTAB data missing - is Internet enabled for git clone?'
print('Repo ready. Data files:', os.listdir('data/MGTAB'))

## 3. Install dependencies
Kaggle ships a GPU build of PyTorch, so we only add the rest.

In [ ]:
!pip install -q "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" "numpy>=1.24"
import sklearn, scipy, pandas
print('sklearn', sklearn.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__)

## 4. Quick smoke test (about 1 minute)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py --mode smoke --task bot --epochs 2 --patience 2 --quiet
print('\nSmoke test finished OK.')

## 5. Full experiment suite
Baselines + hyperparameter tuning + ablation on the bot task across 5 seeds. On a Kaggle
GPU this typically runs in well under an hour and fits comfortably in the background commit.

In [ ]:
OUTDIR = '/kaggle/working/results'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py \
    --mode all \
    --task bot \
    --seeds 42,52,62,72,82 \
    --epochs 500 \
    --patience 50 \
    --trials 12 \
    --output-dir {OUTDIR} \
    --quiet
print('\nFull suite complete.')

## 5b. Adversarial robustness experiment (the novelty)

This is the experiment that matches the paper's "relation camouflage" thesis. It injects
fake&harr;genuine edges into the graph at 0 / 5 / 10 / 20% and measures how far each model's
macro F1 falls. The gated HMR model is designed to suppress such edges, so it should degrade
**least**, while ungated relational baselines (RGCN, directional RGCN) should fall faster. The
MLP is a graph-agnostic control and should stay flat.

In [ ]:
import pandas as pd, os
OUTDIR = '/kaggle/working/results'
def show(path, title):
    if os.path.exists(path):
        print('\n===', title, '===')
        display(pd.read_csv(path))
    else:
        print('(missing)', path)
show(f'{OUTDIR}/bot/tables/baseline_comparison.csv', 'BASELINE COMPARISON (mean +/- std)')
show(f'{OUTDIR}/bot/tables/ablation_comparison.csv', 'ABLATION STUDY')
show(f'{OUTDIR}/bot/tables/significance_tests.csv', 'SIGNIFICANCE TESTS')
show(f'{OUTDIR}/bot/tables/hyperparameter_tuning.csv', 'HYPERPARAMETER TUNING')
show(f'{OUTDIR}/bot/tables/attack_robustness.csv', 'ADVERSARIAL ROBUSTNESS')
show(f'{OUTDIR}/bot/tables/attack_degradation.csv', 'ROBUSTNESS DEGRADATION (smaller drop = more robust)')

## 6. Preview the key paper tables

In [ ]:
import os, glob, shutil

# Find the results directory wherever it ended up, then zip it.
candidates = ['/kaggle/working/results', '/kaggle/working/HMR-GNN/results', './results']
src = next((p for p in candidates if os.path.isdir(p) and os.listdir(p)), None)
if src is None:
    print('NO results found. CSVs under /kaggle/working:')
    print('\n'.join(glob.glob('/kaggle/working/**/tables/*.csv', recursive=True)) or 'nothing found')
else:
    print('Using results dir:', src)
    print('Files inside:', len(glob.glob(os.path.join(src, '**', '*.*'), recursive=True)))
    out = shutil.make_archive('/kaggle/working/hmrgnn_results', 'zip', src)
    print('Wrote:', out, '| size MB:', round(os.path.getsize(out) / 1e6, 2))
    from IPython.display import FileLink
    os.chdir('/kaggle/working')
    display(FileLink('hmrgnn_results.zip'))

## 7. Bundle results into a zip
Everything in `/kaggle/working` is saved as the notebook Output after the commit finishes;
this zip makes it a single convenient download.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/hmrgnn_results', 'zip', '/kaggle/working/results')
print('Wrote /kaggle/working/hmrgnn_results.zip')